In [0]:
%pip install databricks-feature-engineering

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "claims_dev")

In [0]:
import mlflow
from mlflow import MlflowClient
from databricks.feature_engineering import FeatureEngineeringClient
from pyspark.sql import functions as F

catalog = dbutils.widgets.get("catalog")
model_name = f"{catalog}.ml.fraud_model"

mlflow.set_registry_uri("databricks-uc")
version = MlflowClient().get_model_version_by_alias(model_name, "champion").version

claims = spark.table(f"{catalog}.gold.fact_claim").filter("adjusted_ts IS NOT NULL").select("claim_id")
scored = FeatureEngineeringClient().score_batch(model_uri=f"models:/{model_name}@champion", df=claims)

(
    scored.select(
        "claim_id",
        F.col("prediction").cast("int").alias("is_suspicious"),
        F.lit(version).cast("int").alias("model_version"),
        F.current_timestamp().alias("scored_at"),
    )
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.gold.claim_fraud_scores")
)

In [0]:
display(spark.sql(f"""
    SELECT count(*) AS scored, sum(is_suspicious) AS suspicious, max(model_version) AS model_version
    FROM {catalog}.gold.claim_fraud_scores
"""))
display(spark.sql(f"""
    SELECT f.city, count(*) AS claims, sum(s.is_suspicious) AS suspicious
    FROM {catalog}.gold.claim_fraud_scores s
    JOIN {catalog}.gold.fact_claim f ON s.claim_id = f.claim_id
    GROUP BY f.city
    ORDER BY suspicious DESC
"""))